In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
os.environ['GEMINI_API_KEY']=os.getenv("GEMINI_API_KEY")
os.environ['LANGCHAIN_API_KEY']=os.getenv("langchain_api_key")
os.environ['LANGCHAIN_TRACING_V2']="true"
os.environ['LANGCHAIN_PROJECT']=os.getenv("langchain_project")

In [3]:
from langchain_community.document_loaders import WebBaseLoader
loader=WebBaseLoader(web_path="https://en.wikipedia.org/wiki/Shivaji")
docs=loader.load()

c:\Users\Lenovo\OneDrive\Desktop\Lanchain_k\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
from langchain_text_splitters import   RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
texts=text_splitter.split_documents(docs)

In [5]:
from langchain_classic.embeddings import OllamaEmbeddings
embeddings=OllamaEmbeddings(model="nomic-embed-text")


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18428\2422962308.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings=OllamaEmbeddings(model="nomic-embed-text")


In [6]:
from langchain_community.vectorstores import FAISS
db=FAISS.from_documents(texts,embeddings)


In [7]:
result=db.similarity_search("Who is Shivaji?",k=2)

In [8]:
from langchain_ollama import ChatOllama
llm=ChatOllama(model="gemma4:31b-cloud")

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
Answer based only on this context:
{context}

Question: {input}
""")

chain = prompt | llm | StrOutputParser()

In [13]:
chain

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer based only on this context:\n{context}\n\nQuestion: {input}\n'), additional_kwargs={})])
| ChatOllama(output_version=None, model='gemma4:31b-cloud')
| StrOutputParser()

In [ ]:
query="Who is Shivaji?"

In [14]:
retriever = db.as_retriever()
docs = retriever.invoke("Who is Shivaji?")
response = chain.invoke({
    "context": docs,
    "input": "Who is Shivaji?"
})

In [15]:
print(response)

Shivaji was the Chhatrapati of the Marathas from 1674 to 1680.


In [16]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002597F025BA0>, search_kwargs={})

In [17]:
response

'Shivaji was the Chhatrapati of the Marathas from 1674 to 1680.'